# BariatricRSD — Project Walkthrough

**A complete narrative of the experimental work done so far for the NeurIPS 2026 submission.**

**Goal:** A transformer model that predicts Remaining Surgery Duration (RSD), detects intraoperative deviations, and recognizes surgical phases in bariatric (RYGB) surgery videos.

**Deadline:** May 4 (abstract) / May 6 (full paper), 2026 AoE.

**Core hypothesis (H1):** explicit phase-order conditioning (a learnable token encoding the surgeon's procedural style) improves long-horizon prediction over generic temporal models.

This notebook walks through everything — from infrastructure setup to the key experimental findings.

---
## 1. Infrastructure — Lambda Cloud GPU

Training runs happen on a Lambda Cloud GH200 instance (NVIDIA GH200 480GB, 97 GB VRAM, 64 CPU cores). The key layout:

| Path | Purpose |
|------|---------|
| `/home/ubuntu/bariatric-rsd/` | Project root — **symlink to `/lambda/nfs/bariatric-rsd/`** |
| `/lambda/nfs/bariatric-rsd/` | Persistent NFS storage — survives instance termination |
| `/lambda/nfs/bariatric-rsd/extern/` | Public repos (Surgformer, MultiBypass140) |
| `/lambda/nfs/bariatric-rsd/src/` | Project source (model, data, training) |
| `/lambda/nfs/bariatric-rsd/outputs/` | Run outputs, best checkpoints, logs |
| `/lambda/nfs/bariatric-rsd/wandb/` | WandB run data |
| `/lambda/nfs/bariatric-rsd/labels/` | Processed `labels.json` files |

**Why the symlink matters:** Anything under `/home/ubuntu/` is wiped when the instance terminates, but `/lambda/nfs/` persists. Putting everything on NFS means we can stop/start instances without losing work.

In [ ]:
# Connect to Lambda from local — requires ~/.ssh/config entry for `lambda-rsd`
# SSH key: /Users/bill/Documents/GitHub/bariatric_rsd/rsd.pem
!ssh lambda-rsd "nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader" 2>/dev/null

---
## 2. Data Pipeline — MultiBypass140

### Why MultiBypass140?
Public RYGB (Roux-en-Y gastric bypass) benchmark: **140 videos from 2 hospitals** (Bern & Strasbourg, 70 each). Has phase, step, and **intraoperative adverse event (IAE)** annotations. Perfect fit for our multi-task story.

### Download (~365 GB total)
```bash
wget https://s3.unistra.fr/camma_public/datasets/MultiBypass140/multibypass01_corrected.zip  # 86 GB
wget https://s3.unistra.fr/camma_public/datasets/MultiBypass140/multibypass02.zip            # 68 GB
wget https://s3.unistra.fr/camma_public/datasets/MultiBypass140/multibypass03.zip            # 1.2 MB (metadata only)
wget https://s3.unistra.fr/camma_public/datasets/MultiBypass140/multibypass04.zip            # 83 GB
wget https://s3.unistra.fr/camma_public/datasets/MultiBypass140/multibypass05.zip            # 129 GB
wget https://s3.unistra.fr/camma_public/datasets/MultiBypass140/multibypass06_corrected.zip  # 28 MB (IAE labels)
```
Download took ~3 hours. Ran inside `screen` so it survived SSH disconnections.

### Unzip
~77 min total. Videos land in `BernBypass70/videos/BBP*.mp4` and `StrasBypass70/videos/SBP*.mp4`.

### Frame extraction (the clever part)
The default script in MultiBypass140 does this serially — would take ~10 hours for 140 videos. With **16 parallel ffmpeg jobs on 64 CPU cores**, we finished in **12 minutes**.

In [ ]:
# Parallel frame-extraction recipe that actually works on Lambda
# (Run this on the instance; it extracts at 1 fps using 16 parallel ffmpeg processes)
extract_command = r"""
cat > /tmp/extract_one.sh <<'EOF'
#!/bin/bash
vid="$1"; outbase="$2"
name=$(basename "$vid" .mp4)
outdir="$outbase/$name"
mkdir -p "$outdir"
# Skip if already extracted
[ "$(ls -1 "$outdir" | wc -l)" -ge 10 ] && exit 0
ffmpeg -nostdin -loglevel error -y -i "$vid" -vf fps=1 -q:v 2 "$outdir/${name}_%08d.jpg"
EOF
chmod +x /tmp/extract_one.sh

# Run 16 jobs in parallel
ls /path/to/videos/*.mp4 | xargs -n 1 -P 16 -I {} /tmp/extract_one.sh {} /path/to/frames
"""
print(extract_command)

**Resulting frame counts:**
- Bern: 70 videos → **316,643 frames**
- Stras: 70 videos → **464,955 frames**
- **Total: 781,598 frames** at 1 fps

---
## 3. Building `labels.json`

The project's training code (`BariatricFrameDataset`) expects a single `labels.json` with this schema per video:
```python
{
  "video_id": str,
  "total_duration_sec": float,
  "phase_sequence": [list of phase names in order of appearance],
  "phase_vocab": {name: index},
  "phase_order_cluster": int,   # crucial — the paper's key conditioning signal
  "split": "train" | "val" | "test",
  "frames": [
    {"frame_idx": int, "timestamp_sec": float, "rsd_sec": float, "rsd_normalized": float,
     "phase": str, "is_deviation": bool, "frame_path": str}
  ]
}
```

MB140 ships its labels as **per-frame pickles** (one per video) with fields like:
`Frame_id`, `Phase_gt` (int 0-13), `Step_gt`, `Event_ID`, `Overall` (IAE flag), plus 5 IAE categories × 5 severities.

The adapter script [`scripts/06_build_labels.py`](../lambda_setup/scripts/06_build_labels.py) merges these into the expected JSON.

### First fold split (fold 0):
- **80 train** (40 Bern + 40 Stras)
- **20 val** (10 Bern + 10 Stras)
- **40 test** (20 Bern + 20 Stras)

---
## 4. Phase-Order Clustering — the KEY insight

**Phase-order conditioning** is the paper's core contribution. The idea: surgeons have different "styles" in how they sequence the 12 MB140 phases. If we group videos by their phase-transition patterns, the model can condition its predictions on that style.

### Attempt 1 — Heuristic clustering (Run 001)
The project shipped `infer_phase_order_cluster()` — a rule-based mapper using abbreviations like `GC`, `GJ`, `JJ`. Unfortunately, MB140 uses full names like `gastric_pouch_creation`, `gastrojejunal_anastomosis`, so the heuristic doesn't match.

**Result:** 104 of 140 videos land in the `UNKNOWN` cluster (7), 36 in cluster 0. This is basically random — the clustering provides noise, not signal.

### Attempt 2 — Data-driven k-means (Run 006)
Built [`scripts/07_cluster_phase_orders.py`](../lambda_setup/scripts/07_cluster_phase_orders.py):
1. Extract **phase bigrams** (ordered pairs of adjacent phases) for each video
2. TF-IDF over the bigrams (63 unique bigram types across the corpus)
3. PCA → 16 components (76% variance retained)
4. KMeans with k=6

**Result:** 6 balanced, meaningful clusters {38, 27, 25, 23, 17, 10}.

In [ ]:
import json
from collections import Counter

# Load both label files from the local mirror
heuristic = json.load(open('../lambda_mirror/labels/mb140_fold0_labels.json'))
kmeans    = json.load(open('../lambda_mirror/labels/mb140_fold0_labels_kmeans.json'))

print('HEURISTIC clustering (Run 001):')
print('  Cluster sizes:', dict(Counter(v['phase_order_cluster'] for v in heuristic)))

print('\nKMEANS clustering (Run 006):')
print('  Cluster sizes:', dict(Counter(v['phase_order_cluster'] for v in kmeans)))

---
## 5. Model Architecture

`BariatricRSD` (149.9M params, 106.8M trainable with 6 encoder blocks frozen):

```
Video clip (B, T=8, 3, 224, 224)
         ↓
  ViT-Base encoder (timm, 12 blocks, first 6 frozen)
         ↓
  per-frame features (B, T, 768) with dynamic_img_size=True
         ↓
  [phase_order_token] + [frame_0, frame_1, ..., frame_T-1]
         ↓
  Hierarchical Temporal Attention (6 blocks, 12 heads, 3 scales)
         ↓
  Global representation = token at position 0 (phase-order slot)
         ↓
  ┌────────────┬──────────────────┬─────────────┐
  ↓            ↓                  ↓             ↓
 RSD head    Deviation head    Phase head    Operation log
```

**Multi-task loss:** uncertainty-weighted (Kendall et al. 2018):
$$L = \sum_k \exp(-\sigma_k) \cdot L_k + \sigma_k$$
where $\sigma_k$ is a learnable log-variance per task. This explains why total loss can go negative during training — it's a feature, not a bug.

---
## 6. Experimental Chronicle

Eight runs, each testing a specific hypothesis. **Wall time: ~14 hours of training total.**

In [ ]:
import pandas as pd

runs = pd.DataFrame([
    dict(run='001', config='lr=1e-4, freeze=6, multi-task, heuristic clusters',
         purpose='Initial baseline', min=13.20, mean=14.07, std=0.64, verdict='Plateau — heuristic clusters hurt'),
    dict(run='002', config='lr=3e-5, freeze=9, wd=0.1',
         purpose='Stronger regularization', min=14.58, mean=15.27, std=0.62, verdict='Too restrictive'),
    dict(run='003', config='lr=5e-5, freeze=6, wd=0.1',
         purpose='Milder regularization', min=13.29, mean=14.23, std=0.85, verdict='~= Run 001'),
    dict(run='004', config='multi → RSD-only (--disable_dev --disable_phase)',
         purpose='H2: does multi-task hurt RSD?', min=13.60, mean=14.10, std=0.66, verdict='H2 refuted'),
    dict(run='005', config='--no_phase_order (kills the token)',
         purpose='H1 (first attempt)', min=12.51, mean=14.23, std=1.04, verdict='Better but brittle — clusters were broken'),
    dict(run='006', config='kmeans phase-order, full 15 epochs',
         purpose='H1 with proper clusters', min=12.27, mean=12.84, std=0.49, verdict='🏆 BEST on all metrics'),
    dict(run='007', config='kmeans labels + --no_phase_order, full 15 epochs',
         purpose='H1 matched-compute control', min=13.55, mean=14.33, std=0.94, verdict='Loses to Run 006 — H1 validated'),
    dict(run='008', config='train on Bern, val on Stras, kmeans phase-order',
         purpose='Cross-center generalization', min=None, mean=None, std=None, verdict='🔄 running'),
])
runs

### Run 001 — why it crashed

First attempt hit a scipy API mismatch in `compute_rsd_metrics()`: `pearsonr(...).statistic` doesn't exist in older scipy (which returns a tuple). Fixed by switching to `pearsonr(...)[0]`. Took ~10 min to detect and fix.

### Run 001 — what we learned

- **Convergence is FAST:** val_mae hits best (13.20) at **epoch 1**.
- **Train loss plateaus at ~0.002** (nearly solved on train set) but val stays at 13-15 min — classic overfit.
- Per-epoch wall time: ~9-10 min.
- Key suspicion after Run 001: the `phase_order_cluster` signal is broken.

### Runs 002 & 003 — regularization tuning

Tried freezing more of the encoder (to prevent overfit) and lowering LR. Neither broke the 13.20 ceiling. **Lesson:** hyperparameter tuning won't fix a signal quality problem.

### Run 004 — is multi-task the problem?

Patched `train.py` with `--disable_dev --disable_phase` flags to zero the non-RSD loss weights. Single-task RSD got best 13.60 — **not meaningfully better** than multi-task. H2 refuted: **multi-task interference isn't the issue.**

### Run 005 — the surprising result

Patched `train.py` with `--no_phase_order` to force `phase_order_cluster=0` for every sample (collapsing the embedding to a shared CLS token). Expected this to be *worse* than Run 001.

**Instead: val_mae dropped to 12.51 min at epoch 1. Best result yet.**

Why? Because the heuristic clustering was *noise*, and noise as input is worse than no signal. Later epochs oscillated wildly (up to 15.59), confirming the token was noisy rather than informative.

### Run 006 — the real H1 test

Swap in the k-means clustering. Result: **best val_mae = 12.27 min at epoch 13** (cosine LR decay let it keep improving). Mean 12.84, std 0.49. **Beats Run 001 AND Run 005 on all metrics.**

### Run 007 — matched-compute control

Run 005 was killed early (uneven compute budget). Run 007 = same 15 epochs as Run 006 but with `--no_phase_order`. Result: best 13.55, mean 14.33, std 0.94. **Phase-order token with meaningful clusters buys ≈1.3 min improvement and halves variance.**

### Run 008 — the generalization test (running now)

Train on Bern's 70 videos only; evaluate on all 70 Stras videos. Tests whether the model generalizes across hospitals (different surgeons, cameras, workflows). Key number to watch: gap from Run 006's 12.27.

---
## 7. Visualization — the headline finding

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Extracted from the run logs (val_mae per epoch)
run006 = [12.92, 13.99, 12.79, 12.62, 13.29, 13.00, 13.03, 13.65, 12.57, 12.59, 12.34, 12.78, 12.40, 12.27, 12.42]
run007 = [13.55, 17.26, 15.35, 14.15, 13.89, 14.26, 15.01, 13.83, 13.87, 13.93, 13.99, 13.87, 14.28, 13.90, 13.83]

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].plot(run006, 'o-', color='C2', label='Run 006: WITH phase-order (kmeans)')
ax[0].plot(run007, 'x--', color='C3', label='Run 007: NO phase-order (matched control)')
ax[0].axhline(12.27, color='C2', linestyle=':', alpha=0.5, label='Run 006 best (12.27)')
ax[0].axhline(13.55, color='C3', linestyle=':', alpha=0.5, label='Run 007 best (13.55)')
ax[0].set_xlabel('Epoch')
ax[0].set_ylabel('Validation MAE (minutes)')
ax[0].set_title('H1 validated: phase-order conditioning helps')
ax[0].legend()
ax[0].grid(alpha=0.3)

metrics = ['Min', 'Mean', 'Std', 'Max']
vals_006 = [min(run006), np.mean(run006), np.std(run006), max(run006)]
vals_007 = [min(run007), np.mean(run007), np.std(run007), max(run007)]
x = np.arange(len(metrics))
w = 0.35
ax[1].bar(x - w/2, vals_006, w, label='Run 006 (w/ token)', color='C2')
ax[1].bar(x + w/2, vals_007, w, label='Run 007 (no token)', color='C3')
ax[1].set_xticks(x)
ax[1].set_xticklabels(metrics)
ax[1].set_ylabel('val_mae (minutes)')
ax[1].set_title('Metric-level comparison')
ax[1].legend()
ax[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../lambda_mirror/h1_result.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Key findings summary

| Finding | Evidence | Paper implication |
|---------|----------|-------------------|
| **H1 VALIDATED** — phase-order conditioning (with data-driven clusters) improves RSD | Run 006 vs Run 007, matched compute: -1.28 min min, -1.49 min mean, halved std | **Paper headline** |
| **Clustering quality matters decisively** | Run 001 (heuristic) vs Run 006 (kmeans): 13.20 → 12.27 min | Methods section must detail clustering |
| **H2 REFUTED** — multi-task doesn't hurt RSD | Run 004 ≈ Run 001 on RSD | Multi-task is "free" — keep it |
| **Phase-order provides training stability** | Run 006 std=0.49 vs Run 007 std=0.94 | Robustness/reliability narrative |
| **Training converges in ~1 epoch, needs LR decay for final push** | Run 006 best at epoch 13 with cosine decay | Hyperparameter findings for appendix |

---
## 9. Code changes shipped

All changes are on the Lambda instance; patched files live in `lambda_mirror/src_patched/` after running `scripts/sync_from_lambda.sh`.

### `src/training/train.py`
- **Bug fix** (line 55-56): scipy API compatibility — `pearsonr(...).statistic` → `pearsonr(...)[0]`
- **New CLI flags:**
  - `--disable_dev` — zeros deviation loss weight
  - `--disable_phase` — zeros phase loss weight
  - `--no_phase_order` — forces `phase_order_cluster=0` for every sample

### `src/models/bariatric_rsd.py`
- **Bug fix**: added `dynamic_img_size=True` to `timm.create_model()` — required because the smoke test uses 64×64 inputs.

### New scripts
- `lambda_setup/scripts/06_build_labels.py` — builds `labels.json` from the MB140 pickle files
- `lambda_setup/scripts/07_cluster_phase_orders.py` — k-means on phase-bigram TF-IDF
- `scripts/sync_from_lambda.sh` — pulls essential artifacts to local

---
## 10. Next actions

1. **Run 008 (running)** — cross-center Bern→Stras generalization (confirms paper's workflow-variability narrative)
2. **Run 009** — phase-order cluster count sensitivity (k=4, 6, 8) for the ablation appendix
3. **Run 010** — 3-seed replication of Run 006 for statistical significance (paper requires this)
4. **Test-set evaluation** — Run 006's best checkpoint on the 40 held-out test videos (not val)
5. **Cholec80 reproduction** — establish we can match published numbers on an easier benchmark
6. **Paper writing** — port content from `GPT_plan.md` outline into the NeurIPS LaTeX template

Deadlines (from 2026-04-22):
- **May 4 AoE** — abstract (12 days)
- **May 6 AoE** — full paper (14 days)

---
## Appendix — operational commands

```bash
# Check what's running on Lambda
ssh lambda-rsd "screen -ls && pgrep -af 'train.py'"

# Tail a live training log
ssh lambda-rsd "tail -f /tmp/run008.log"

# See all validation results for a run
ssh lambda-rsd "grep -E 'Epoch +[0-9]+ \\| val' /tmp/run008.log"

# Pull artifacts to local
bash scripts/sync_from_lambda.sh

# Preview what would be synced without transferring
bash scripts/sync_from_lambda.sh --dry-run
```